# 02_pipeline — governed pipeline output orchestration template

Read source data, register DataFrames, profile data, transform into pipeline outputs, optionally enrich metadata and author guardrails through widgets, enforce guardrails, then write pipeline outputs and runtime metadata.

Only edit the source reads, transformations, output write settings, and lineage relationships. Schema, freshness, profile, DQ guardrail authoring, and enrichment are handled by widgets after profiles exist.


## 1. Run `00_env_config`


In [ ]:
%run 00_env_config


In [ ]:
# ============================================================
# Notebook display settings
# Usually no change needed
# ============================================================

# display_guardrail_results defaults to summary mode. Set this only when you need detailed/debug output.
GUARDRAIL_DISPLAY_MODE = "summary"


## 2. Import required functions


In [ ]:
from pyspark.sql import functions as F

from fabricops_kit import (
    display_guardrail_results,
    prepare_pipeline_table_configs,
    read_lakehouse_csv,
    read_lakehouse_excel,
    read_lakehouse_parquet,
    read_lakehouse_table,
    read_warehouse_query,
    read_warehouse_table,
    run_table_guardrails,
    start_pipeline_run,
    widget_author_dq_rules,
    widget_author_schema_freshness_profile_rules,
    widget_enrich_table_metadata,
    widget_review_guardrail_governance,
    widget_select_guardrail_target,
    write_lakehouse_table,
    write_warehouse_table,
    write_pipeline_lineage,
    write_pipeline_run_summary,
)


## 3. Select agreement and capture run context


In [ ]:
PIPELINE = start_pipeline_run(
    notebook_type="02_pipeline",
    select_agreement=True,
    register_notebook=True,
)


---
# SOURCE AREA

Read source data, register key + DataFrame only, then profile data.


## 4. USER EDIT SECTION — read source DataFrames

The read call carries the physical source identity. Registration only needs a stable key and the DataFrame object.


In [ ]:
# ============================================================
# User inputs
# Change this section for your use case
# ============================================================
source_table = "demo_src_orders_happy"  # Change this to your primary input table
customer_table = "demo_src_customers_happy"  # Change this to your secondary input table, if needed

df_orders = read_lakehouse_table(source_table, target="source", schema="DemoTest", spark_session=spark)

df_customers = read_lakehouse_table(customer_table, target="source", schema="DemoTest", spark_session=spark)

# Demo defaults: "demo_src_orders_happy" and "demo_src_customers_happy".

# Other examples:
# df_orders = read_lakehouse_csv("path/to/orders.csv", spark_session=spark, header=True)
# df_orders = read_lakehouse_parquet("path/to/orders.parquet", spark_session=spark)
# df_customers = read_lakehouse_excel("path/to/customers.xlsx", sheet_name=0, spark_session=spark)
# df_orders = read_warehouse_query("SELECT order_id, customer_id, status FROM dbo.orders WHERE status = 'OPEN'", target="warehouse", spark_session=spark)


## 5. USER EDIT SECTION — register source DataFrames only

Do not define schema, freshness, profile behaviour, DQ, classification, enrichment, or distribution settings here. Those are profiled and curated through widgets.


In [ ]:
SOURCE_TABLES = [
    {"key": "orders", "df": df_orders},
    {"key": "customers", "df": df_customers},
]

SOURCE_TABLES, SOURCE_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    SOURCE_TABLES,
    {},
    table_role="source",
)

df_orders = SOURCE_CONFIG_BY_KEY["orders"]["df"]
df_customers = SOURCE_CONFIG_BY_KEY["customers"]["df"]


## 6. Profile source data

This records source-side profiles in `METADATA_DATA_CATALOGUE`. It is profile generation, not manual schema authoring.


In [ ]:
source_profile_results = run_table_guardrails(
    SOURCE_TABLES,
    table_role="source",
    mode="profile",
)

display_guardrail_results(source_profile_results)


---
# TRANSFORMATION AND TARGET AREA


## 7. USER EDIT SECTION — transform source DataFrames into pipeline outputs


In [ ]:
df_orders_enriched = (
    df_orders.alias("o")
    .join(df_customers.alias("c"), on="customer_id", how="left")
    .select(
        "order_id", "customer_id", "customer_name", "customer_segment",
        F.col("country_code").alias("order_country_code"),
        F.col("customer_country_code"), "order_date", "status", "order_amount",
        F.current_timestamp().alias("processed_ts"),
    )
)

df_orders_summary = (
    df_orders_enriched
    .groupBy("order_date", "customer_segment", "status")
    .agg(F.count("order_id").alias("order_count"), F.sum("order_amount").alias("total_order_amount"))
)


## 8. USER EDIT SECTION — register pipeline outputs only

Pipeline outputs can be profiled before physical output tables exist because the DataFrame already exists.


In [ ]:
TARGET_TABLES = [
    {"key": "orders_enriched", "df": df_orders_enriched},
    {"key": "orders_summary", "df": df_orders_summary},
]

TARGET_TABLES, TARGET_CONFIG_BY_KEY = prepare_pipeline_table_configs(
    TARGET_TABLES,
    {},
    table_role="target",
    run_id=PIPELINE.run_id,
    pipeline_name=PIPELINE.pipeline_name,
)

df_orders_enriched = TARGET_CONFIG_BY_KEY["orders_enriched"]["df"]
df_orders_summary = TARGET_CONFIG_BY_KEY["orders_summary"]["df"]


## 9. Profile pipeline outputs


In [ ]:
target_profile_results = run_table_guardrails(
    TARGET_TABLES,
    table_role="target",
    mode="profile",
)

display_guardrail_results(target_profile_results)


## 10. Optional enrichment and guardrail widgets

Run only when you need to author or review schema/freshness/profile/DQ rules or enrich metadata from the latest profiles.


In [ ]:
selected_guardrail_target = widget_select_guardrail_target(spark_session=spark)

widget_author_schema_freshness_profile_rules(
    selected_guardrail_target,
    spark_session=spark,
)
widget_author_dq_rules(
    selected_guardrail_target,
    spark_session=spark,
)
widget_enrich_table_metadata(
    selected_guardrail_target,
    spark_session=spark,
)
widget_review_guardrail_governance(
    selected_guardrail_target,
    spark_session=spark,
)


## 11. Guardrail enforcement gate

If any blocking source or pipeline output guardrail fails, the notebook stops here before output write settings and before any output table write.


In [ ]:
source_enforcement_results = run_table_guardrails(
    SOURCE_TABLES,
    table_role="source",
    mode="enforce",
)
display_guardrail_results(source_enforcement_results)
# run_table_guardrails stops the notebook when blocking source guardrails fail.

target_enforcement_results = run_table_guardrails(
    TARGET_TABLES,
    table_role="target",
    mode="enforce",
)
display_guardrail_results(target_enforcement_results)
# run_table_guardrails stops the notebook when blocking target guardrails fail.


## 12. USER EDIT SECTION — configure output write settings

Write settings belong after the guardrail gate. `target_name` and `write_mode` are essential. Layer, schema, options, partitioning, and repartitioning are write concerns only.


In [ ]:
TARGET_WRITE_SETTINGS = {
    "orders_enriched": {
        "target_layer": "unified",
        "target_name": "demo_unified_orders_enriched",
        "schema": "DemoTest",
        "write_mode": "overwrite",
        "options": {"overwriteSchema": "true"},
    },
    "orders_summary": {
        "target_layer": "unified",
        "target_name": "demo_unified_orders_summary",
        "schema": "DemoTest",
        "write_mode": "overwrite",
        "options": {"overwriteSchema": "true"},
        # Optional: "partition_by": ["order_date"],
        # Optional: "repartition_by": ["customer_segment"],
    },
}

for key, write_settings in TARGET_WRITE_SETTINGS.items():
    TARGET_CONFIG_BY_KEY[key].update(write_settings)


## 13. Write pipeline output Lakehouse tables

This section only runs after all blocking guardrails pass.


In [ ]:
target_write_status = {}

for key, target in TARGET_CONFIG_BY_KEY.items():
    write_lakehouse_table(
        target["df"],
        target["target_name"],
        target=target.get("target_layer", "unified"),
        schema=target.get("schema"),
        mode=target.get("write_mode", "overwrite"),
        partition_by=target.get("partition_by"),
        repartition_by=target.get("repartition_by"),
        options=target.get("options"),
    )
    target_write_status[key] = f"written: {target.get('schema')}.{target['target_name']}"

target_write_status


## 14. Optional warehouse write example


In [ ]:
# orders_summary_target = TARGET_CONFIG_BY_KEY["orders_summary"]
# write_lakehouse_table(
#     orders_summary_target["df"], orders_summary_target["target_name"],
#     target="product", format="warehouse", schema="dbo",
#     mode=orders_summary_target.get("write_mode", "overwrite"),
# )


## 15. USER EDIT SECTION — lineage relationships


In [ ]:
LINEAGE_RELATIONSHIPS = [
    {"source_key": "orders", "target_key": "orders_enriched", "transformation_type": "join", "transformation_logic": "orders joined to customers on customer_id"},
    {"source_key": "customers", "target_key": "orders_enriched", "transformation_type": "join", "transformation_logic": "customers joined to orders on customer_id"},
    {"source_key": "orders_enriched", "target_key": "orders_summary", "transformation_type": "aggregation", "transformation_logic": "group by order_date, customer_segment, and status"},
]


## 16. Write lineage metadata


In [ ]:
lineage_result = write_pipeline_lineage(
    spark=spark,
    run_id=PIPELINE.run_id,
    source_definitions=SOURCE_CONFIG_BY_KEY,
    target_definitions=TARGET_CONFIG_BY_KEY,
    relationships=LINEAGE_RELATIONSHIPS,
    pipeline_name=PIPELINE.pipeline_name,
    notebook_id=PIPELINE.notebook_id,
    notebook_registry_id=PIPELINE.notebook_registry_id,
    agreement_id=PIPELINE.agreement_id,
    agreement_contract_version=PIPELINE.agreement_contract_version,
)
lineage_result


## 17. Write runtime summary


In [ ]:
runtime_summary_result = write_pipeline_run_summary(
    source_guardrail_results=source_enforcement_results,
    target_guardrail_results=target_enforcement_results,
    target_write_status=target_write_status,
    lineage_result=lineage_result,
)
runtime_summary_result
